In [1]:
import json
import sqlite3
from pathlib import Path

# ---- CONFIG ----
DATASET_ROOT = Path(".")          # set to Path("/path/to/2026-01-27_201419_domain_with_snapshots") if needed
METADATA_DIR = DATASET_ROOT / "metadata"

N_FILES = 1000                    # set to None for all metadata shards
MIN_SNAPSHOTS = 2                 # snapshots_metadata threshold
MIN_FILTERED = 4                  # filtered_snapshots_metadata threshold

DB_PATH = DATASET_ROOT / "group_counts.sqlite"

OUT_SNAPSHOTS_JSONL = DATASET_ROOT / "snapshots_metadata.jsonl"
OUT_FILTERED_JSONL  = DATASET_ROOT / "filtered_snapshots_metadata.jsonl"

In [2]:
def iter_metadata_shards(metadata_dir: Path, n_files=None):
    shards = sorted(metadata_dir.glob("*.jsonl"))
    return shards if n_files is None else shards[:n_files]

def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return str(x)

def group_key_from_record(rec):
    # CFLW equivalent of (path, page_title) => (domain_id, path, title)
    return (
        safe_str(rec.get("domain_id")),
        safe_str(rec.get("path")),
        safe_str(rec.get("title")),
    )

def is_valid_snapshot(rec, dataset_root: Path) -> bool:
    rel = rec.get("html_file_location")
    if not rel:
        return False
    return (dataset_root / rel).exists()

def init_db(db_path: Path):
    conn = sqlite3.connect(str(db_path))
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS group_counts (
            domain_id TEXT NOT NULL,
            path      TEXT NOT NULL,
            title     TEXT NOT NULL,
            cnt       INTEGER NOT NULL,
            PRIMARY KEY (domain_id, path, title)
        )
    """)
    return conn

In [3]:
def pass1_count_groups(dataset_root: Path, metadata_dir: Path, db_path: Path, n_files=None):
    shards = iter_metadata_shards(metadata_dir, n_files=n_files)
    if not shards:
        raise RuntimeError(f"No metadata shards found in {metadata_dir}")

    conn = init_db(db_path)
    cur = conn.cursor()

    upsert_sql = """
        INSERT INTO group_counts(domain_id, path, title, cnt)
        VALUES (?, ?, ?, 1)
        ON CONFLICT(domain_id, path, title) DO UPDATE SET cnt = cnt + 1
    """

    total_lines = 0
    valid_lines = 0
    bad_json = 0

    for shard in shards:
        batch = []
        with shard.open("r", encoding="utf-8", errors="replace") as f:
            for line in f:
                total_lines += 1
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    bad_json += 1
                    continue

                if not is_valid_snapshot(rec, dataset_root):
                    continue

                valid_lines += 1
                key = group_key_from_record(rec)
                batch.append(key)

                if len(batch) >= 5000:
                    cur.executemany(upsert_sql, batch)
                    conn.commit()
                    batch.clear()

        if batch:
            cur.executemany(upsert_sql, batch)
            conn.commit()

    conn.close()
    print("Pass 1 done.")
    print(f"  Shards processed: {len(shards)}")
    print(f"  Total lines seen: {total_lines:,}")
    print(f"  Valid snapshots counted: {valid_lines:,}")
    print(f"  Bad JSON skipped: {bad_json:,}")
    print(f"  DB saved at: {db_path.resolve()}")

In [4]:
def get_group_count(conn, key):
    cur = conn.cursor()
    cur.execute(
        "SELECT cnt FROM group_counts WHERE domain_id=? AND path=? AND title=?",
        key
    )
    row = cur.fetchone()
    return int(row[0]) if row else 0


def pass2_write_outputs_jsonl(
    dataset_root: Path,
    metadata_dir: Path,
    db_path: Path,
    out_snapshots_jsonl: Path,
    out_filtered_jsonl: Path,
    n_files=None,
    min_snapshots=2,
    min_filtered=4
):
    shards = iter_metadata_shards(metadata_dir, n_files=n_files)
    if not shards:
        raise RuntimeError(f"No metadata shards found in {metadata_dir}")

    conn = sqlite3.connect(str(db_path))

    # group_number assignment + snapshot counters
    group_number_map = {}
    snapshot_counter_map = {}
    next_group_number = 1

    wrote_snapshots = 0
    wrote_filtered = 0
    bad_json = 0
    missing_html = 0

    out_snapshots_jsonl.parent.mkdir(parents=True, exist_ok=True)
    out_filtered_jsonl.parent.mkdir(parents=True, exist_ok=True)

    with out_snapshots_jsonl.open("w", encoding="utf-8") as f_all, \
         out_filtered_jsonl.open("w", encoding="utf-8") as f_filt:

        for shard in shards:
            with shard.open("r", encoding="utf-8", errors="replace") as fin:
                for line in fin:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                    except json.JSONDecodeError:
                        bad_json += 1
                        continue

                    # Only keep snapshots with existing HTML file
                    if not is_valid_snapshot(rec, dataset_root):
                        missing_html += 1
                        continue

                    key = group_key_from_record(rec)
                    cnt = get_group_count(conn, key)

                    # Output 1: groups must have >=2 valid snapshots
                    if cnt < min_snapshots:
                        continue

                    # Assign group_number if first time seeing this group
                    if key not in group_number_map:
                        group_number_map[key] = next_group_number
                        snapshot_counter_map[key] = 1
                        next_group_number += 1

                    group_no = group_number_map[key]
                    snap_no = snapshot_counter_map[key]
                    snapshot_counter_map[key] += 1

                    # Add fields (do not remove anything else)
                    rec["group_number"] = group_no
                    rec["snapshot_counter"] = snap_no
                    rec["website_id"] = f"{group_no}_{snap_no}"
                    rec["group_key"] = {"domain_id": key[0], "path": key[1], "title": key[2]}
                    rec["valid_group_snapshot_count"] = cnt

                    line_out = json.dumps(rec, ensure_ascii=False)

                    f_all.write(line_out + "\n")
                    wrote_snapshots += 1

                    # Output 2: also require >=4 valid snapshots
                    if cnt >= min_filtered:
                        f_filt.write(line_out + "\n")
                        wrote_filtered += 1

    conn.close()
    print("Pass 2 done.")
    print(f"  Wrote snapshots JSONL (>= {min_snapshots}/group): {wrote_snapshots:,}")
    print(f"  Wrote filtered JSONL (>= {min_filtered}/group): {wrote_filtered:,}")
    print(f"  Skipped missing html_file_location / missing file: {missing_html:,}")
    print(f"  Bad JSON skipped: {bad_json:,}")
    print(f"  Output 1: {out_snapshots_jsonl.resolve()}")
    print(f"  Output 2: {out_filtered_jsonl.resolve()}")

In [5]:
pass1_count_groups(
    dataset_root=DATASET_ROOT,
    metadata_dir=METADATA_DIR,
    db_path=DB_PATH,
    n_files=N_FILES
)

pass2_write_outputs_jsonl(
    dataset_root=DATASET_ROOT,
    metadata_dir=METADATA_DIR,
    db_path=DB_PATH,
    out_snapshots_jsonl=OUT_SNAPSHOTS_JSONL,
    out_filtered_jsonl=OUT_FILTERED_JSONL,
    n_files=N_FILES,
    min_snapshots=MIN_SNAPSHOTS,
    min_filtered=MIN_FILTERED
)

Pass 1 done.
  Shards processed: 256
  Total lines seen: 11,403,638
  Valid snapshots counted: 11,403,638
  Bad JSON skipped: 0
  DB saved at: /home/darknet/2026-01-27_201419_domain_with_snapshots/group_counts.sqlite
Pass 2 done.
  Wrote snapshots JSONL (>= 2/group): 10,680,393
  Wrote filtered JSONL (>= 4/group): 7,941,221
  Skipped missing html_file_location / missing file: 0
  Bad JSON skipped: 0
  Output 1: /home/darknet/2026-01-27_201419_domain_with_snapshots/snapshots_metadata.jsonl
  Output 2: /home/darknet/2026-01-27_201419_domain_with_snapshots/filtered_snapshots_metadata.jsonl
